## Decision Graph
### Here we decide if a question needs retrieval or not

In [5]:
#dependencies
from typing import List, TypedDict, Literal
from pydantic import BaseModel, Field
import time
import os
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_google_genai import  ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
docs = (PyPDFLoader("./documents/Company_Policies.pdf").load()+ 
        PyPDFLoader("./documents/Company_Profile.pdf").load()+
        PyPDFLoader("./documents/Product_and_Pricing.pdf").load())

In [7]:
primary = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0)
fallback = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

llm = primary.with_fallbacks([fallback])

### Chunking and embedding

In [8]:
chunks = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=150
).split_documents(docs)

In [9]:
embeddings = OllamaEmbeddings(model="mxbai-embed-large")

persist_directory = "./chroma_db"

if os.path.exists(persist_directory):
    print("Loading existing vector database...")
    vector_store = Chroma(
        persist_directory=persist_directory,
        embedding_function=embeddings,
        collection_name="company_knowledge_base"
    )
else:
    print("Creating embeddings for the first time...")
    vector_store = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_name="company_knowledge_base"
    )

Creating embeddings for the first time...
